In [ ]:
import pandas as pd
import requests
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

##Environment Setup
The following commands ensure your Colab environment is ready for using the LLaMA 3.1 model locally through Ollama.

In [ ]:
!sudo apt-get update
!sudo apt-get install -y curl
!curl https://ollama.ai/install.sh | sh

import subprocess
import time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)  # Wait for the server to start

!ollama pull llama3.1:8b

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,724 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [8,992 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-sec

##Mount Google Drive and Load Dataset


In [ ]:
# Mount Google Drive to access files
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Set the path to your dataset in Google Drive
file_path = "/content/drive/MyDrive/Train_data.csv"  # Adjust this path if needed

# Load the dataset into a DataFrame
df = pd.read_csv(file_path)

# Display the first few rows to check the data
print(f"Cleaned dataset shape: {df.shape}")
df.head()

Cleaned dataset shape: (1200, 3)


,Unnamed: 0,label,text
0,0,Psoriasis,I have been experiencing a skin rash on my arm...
1,1,Psoriasis,"My skin has been peeling, especially on my kne..."
2,2,Psoriasis,I have been experiencing joint pain in my fing...
3,3,Psoriasis,"There is a silver like dusting on my skin, esp..."
4,4,Psoriasis,"My nails have small dents or pits in them, and..."


##Noise Generation for Symptom Description Dataset Using LLaMA 3.1

This section enriches the original dataset by generating noisy symptom descriptions using LLaMA 3.1 through Ollama API. This simulates realistic elderly speech patterns with two defined noise levels:


*   Medium Noise: Short anecdotes, mild confusion (~50–200 words).
*   Heavy Noise: Extended confusion, repetitions, false memories (~150–400 words).

This process helps evaluate how robust classification models are under noisy, realistic conditions.

In [ ]:
# General configuration
INPUT_CSV   = "/content/drive/MyDrive/Train_data.csv"
OUTPUT_CSV  = "/content/drive/MyDrive/Train_data_with_noise2.csv"
CHUNK_SIZE  = 50 # Number of rows to load per chunk
MAX_WORKERS = 4 # Number of threads for parallel processing
SLEEP_MS    = 0.05   # Minimum delay between requests (50 ms)

# Initialize a persistent HTTP session to reuse connections
session = requests.Session()

# Noise function
def generate_patient_noise(text: str, level: str) -> str:
  # Build the base prompt enforcing only the patient’s monologue
    base = (
        "Rewrite the following patient symptom description exactly as if told by an elderly patient "
        "speaking to their doctor. Preserve every medical detail, but add background noise that simulates "
        "an older person’s speech (hesitations, brief ramblings, confusion, tangents).\n\n"
        "IMPORTANT: Output only the patient's monologue. Do NOT include any doctor's voice or questions.\n\n"
        "Original description:\n" + text + "\n\n"
    )
    # Append level-specific instructions
    if level == 'medium':
        prompt = base + (
            "— Add medium noise: a few brief anecdotes or off-topic remarks, "
            "light confusion over timing or names, filler words, it has to be total ~50-200 words.\n"
            "Output only the patient's own words."
        )
    else:
        prompt = base + (
            "— Add heavy noise: extended ramblings or repeated phrases, more confusion and tangents, "
            "filler words, occasional false memories, it has to be total ~150-400 words.\n"
            "Output only the patient's own words."
        )
    # Send prompt to local inference API
    res = session.post(
        "http://localhost:11434/api/generate",
        json={"model": "llama3.1:8b", "prompt": prompt, "stream": False}
    )
    time.sleep(SLEEP_MS)
    return res.json().get("response", "").strip()

# Process a single DataFrame chunk, adding two noise columns
def process_chunk(df_chunk: pd.DataFrame) -> pd.DataFrame:
    texts = df_chunk["text"].tolist()
    # Generate medium-noise versions
    df_chunk["medium_noise"] = [
        generate_patient_noise(t, 'medium') for t in texts
    ]
    # Generate heavy-noise versions
    df_chunk["heavy_noise"] = [
        generate_patient_noise(t, 'heavy') for t in texts
    ]
    return df_chunk

# Read the CSV in chunks, process in parallel, and append to output
reader = pd.read_csv(INPUT_CSV, chunksize=CHUNK_SIZE)
futures = []
results = []
first_write = True

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    for chunk in reader:
        futures.append(executor.submit(process_chunk, chunk))

    for future in as_completed(futures):
        df_out = future.result()
        # Write (or append) to output CSV to avoid high memory usage
        if first_write:
            df_out.to_csv(OUTPUT_CSV, index=False, mode='w')
            first_write = False
        else:
            df_out.to_csv(OUTPUT_CSV, index=False, mode='a', header=False)
        start, stop = df_out.index.start, df_out.index.stop
        print(f"Finished rows {start}–{stop}")

print(f"All done! Noisy dataset saved to: {OUTPUT_CSV}")

Finished rows 150–200
Finished rows 0–50
Finished rows 100–150
Finished rows 50–100
Finished rows 200–250
Finished rows 300–350
Finished rows 250–300
Finished rows 350–400
Finished rows 400–450
Finished rows 450–500
Finished rows 500–550
Finished rows 550–600
Finished rows 600–650
Finished rows 650–700
Finished rows 700–750
Finished rows 750–800
Finished rows 800–850
Finished rows 850–900
Finished rows 900–950
Finished rows 950–1000
Finished rows 1050–1100
Finished rows 1000–1050
Finished rows 1100–1150
Finished rows 1150–1200
All done! Noisy dataset saved to: /content/drive/MyDrive/Train_data_with_noise2.csv
